# Text-to-SQL Fine-Tuning with Qwen 2.5-3B + LoRA on Apple MLX

End-to-end notebook covering: data prep → LoRA training → model fusion → 
evaluation (Exact Match, BLEU, Keyword Overlap) → Gradio demo.

See README.md for results and design decisions.

In [ ]:
# ============================================================
# CELL 1: Load & Explore Dataset
# ============================================================
from datasets import load_dataset

ds = load_dataset("gretelai/synthetic_text_to_sql")
print(f"Train: {len(ds['train'])} | Test: {len(ds['test'])}")
print(f"\nColumns: {ds['train'].column_names}")
print(f"\n--- Sample ---")
sample = ds['train'][0]
for k in ['sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'sql_complexity']:
    print(f"\n[{k}]:\n{sample[k]}")


In [ ]:
# ============================================================
# CELL 2: Prepare Training Data (JSONL for MLX)
# ============================================================
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

def format_sample(sample):
    user_msg = f"""Database Schema:
{sample['sql_context']}

Question: {sample['sql_prompt']}"""

    assistant_msg = f"""SQL: {sample['sql']}
Explanation: {sample['sql_explanation']}"""

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
    }

train_data = ds['train'].shuffle(seed=42).select(range(5000))
val_data = ds['test'].shuffle(seed=42).select(range(500))
test_data = ds['test'].shuffle(seed=42).select(range(500, 1000))

for split_name, split_data in [("train", train_data), ("valid", val_data), ("test", test_data)]:
    output_path = DATA_DIR / f"{split_name}.jsonl"
    with open(output_path, "w") as f:
        for s in split_data:
            f.write(json.dumps(format_sample(s)) + "\n")
    print(f"Saved {len(split_data)} samples to {output_path}")

# Sanity check
with open(DATA_DIR / "train.jsonl") as f:
    first = json.loads(f.readline())
print("\n--- First training sample preview ---")
for msg in first["messages"]:
    print(f"\n[{msg['role']}]:\n{msg['content'][:200]}")

In [ ]:
# ============================================================
# CELL 3: LoRA Config
# ============================================================
import yaml

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
ADAPTER_DIR = "adapters"
FUSED_DIR = "fused_model"

lora_config = {
    "model": MODEL_NAME,
    "train": True,
    "data": str(DATA_DIR),
    "seed": 42,
    "lora_layers": 16,
    "batch_size": 1,
    "iters": 1000,
    "val_batches": 25,
    "learning_rate": 1e-5,
    "steps_per_report": 50,
    "steps_per_eval": 100,
    "adapter_path": ADAPTER_DIR,
    "save_every": 200,
    "max_seq_length": 1024,
}

config_path = "lora_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(lora_config, f)

print(f"Config saved to {config_path}")
print(f"Model: {MODEL_NAME}")
print(f"Iterations: {lora_config['iters']}")

In [ ]:
# ============================================================
# CELL 4: Run LoRA Fine-Tuning (~30-60 min on M4)
# ============================================================
import subprocess

print("Starting fine-tuning...")
result = subprocess.run(
    ["python", "-m", "mlx_lm.lora", "--config", config_path],
    capture_output=False, text=True
)
print("Fine-tuning complete!")

In [ ]:
# ============================================================
# CELL 5: Fuse LoRA Adapters into Model
# ============================================================
import subprocess

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
ADAPTER_DIR = "adapters"
FUSED_DIR = "fused_model"

subprocess.run([
    "python", "-m", "mlx_lm.fuse",
    "--model", MODEL_NAME,
    "--adapter-path", ADAPTER_DIR,
    "--save-path", FUSED_DIR
], capture_output=False, text=True)

print(f"Fused model saved to {FUSED_DIR}/")


In [ ]:
# ============================================================
# CELL 6: Quick Test Fine-Tuned Model
# ============================================================
from mlx_lm import load, generate

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

FUSED_DIR = "fused_model"
model, tokenizer = load(FUSED_DIR)

test_prompt = """Database Schema:
CREATE TABLE employees (id INT, name VARCHAR(100), department VARCHAR(50), salary DECIMAL(10,2));
CREATE TABLE departments (id INT, name VARCHAR(50), budget DECIMAL(12,2));

Question: What is the average salary by department?"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
response = generate(model, tokenizer, prompt=prompt, max_tokens=256)
print("--- Fine-tuned Model Output ---")
print(response)

# Free memory
del model, tokenizer

In [ ]:
# ============================================================
# CELL 7: Evaluation — Before vs After (using MLX directly)
# ============================================================
import json
import re
from pathlib import Path
from mlx_lm import load, generate

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

DATA_DIR = Path("data")
BASE_MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
FUSED_DIR = "fused_model"

def mlx_generate(model, tokenizer, user_msg, system=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=300)

def extract_sql(response_text):
    lines = response_text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        if line.strip().upper().startswith("SQL:"):
            capture = True
            sql_lines.append(line.split(":", 1)[1].strip())
        elif line.strip().upper().startswith("EXPLANATION:"):
            capture = False
        elif capture:
            sql_lines.append(line.strip())
    if sql_lines:
        return " ".join(sql_lines).strip().rstrip(";") + ";"
    for line in lines:
        if any(line.strip().upper().startswith(kw) for kw in ["SELECT", "INSERT", "UPDATE", "DELETE", "CREATE", "WITH"]):
            return line.strip()
    return response_text.strip()

def normalize_sql(sql):
    sql = sql.lower().strip().rstrip(";") + ";"
    sql = re.sub(r'\s+', ' ', sql)
    return sql

# Load test data
test_samples = []
with open(DATA_DIR / "test.jsonl") as f:
    for line in f:
        test_samples.append(json.loads(line))

NUM_EVAL = 50

# --- Evaluate Base Model ---
print("Loading base model...")
base_model, base_tokenizer = load(BASE_MODEL_NAME)

base_results = []
print(f"\nEvaluating BASE model on {NUM_EVAL} samples...")
for i, sample in enumerate(test_samples[:NUM_EVAL]):
    user_msg = sample["messages"][1]["content"]
    ground_truth = sample["messages"][2]["content"]
    gt_sql = ""
    for line in ground_truth.split("\n"):
        if line.startswith("SQL:"):
            gt_sql = line.split(":", 1)[1].strip()
            break

    resp = mlx_generate(base_model, base_tokenizer, user_msg)
    pred_sql = extract_sql(resp)
    match = normalize_sql(pred_sql) == normalize_sql(gt_sql)
    base_results.append({
        "match": match, "pred": pred_sql, "gt": gt_sql,
        "question": user_msg, "full_response": resp
    })
    if (i + 1) % 10 == 0:
        correct = sum(r["match"] for r in base_results)
        print(f"  [{i+1}/{NUM_EVAL}] Base accuracy: {correct}/{i+1}")

del base_model, base_tokenizer
print("Base model unloaded.\n")

# --- Evaluate Fine-Tuned Model ---
print("Loading fine-tuned model...")
ft_model, ft_tokenizer = load(FUSED_DIR)

ft_results = []
print(f"\nEvaluating FINE-TUNED model on {NUM_EVAL} samples...")
for i, sample in enumerate(test_samples[:NUM_EVAL]):
    user_msg = sample["messages"][1]["content"]
    ground_truth = sample["messages"][2]["content"]
    gt_sql = ""
    for line in ground_truth.split("\n"):
        if line.startswith("SQL:"):
            gt_sql = line.split(":", 1)[1].strip()
            break

    resp = mlx_generate(ft_model, ft_tokenizer, user_msg)
    pred_sql = extract_sql(resp)
    match = normalize_sql(pred_sql) == normalize_sql(gt_sql)
    ft_results.append({
        "match": match, "pred": pred_sql, "gt": gt_sql,
        "question": user_msg, "full_response": resp
    })
    if (i + 1) % 10 == 0:
        correct = sum(r["match"] for r in ft_results)
        print(f"  [{i+1}/{NUM_EVAL}] Fine-tuned accuracy: {correct}/{i+1}")

del ft_model, ft_tokenizer
print("Fine-tuned model unloaded.\n")

# --- Results Summary ---
base_acc = sum(r["match"] for r in base_results) / NUM_EVAL * 100
ft_acc = sum(r["match"] for r in ft_results) / NUM_EVAL * 100

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Base Model:       {base_acc:.1f}% exact match ({int(base_acc*NUM_EVAL/100)}/{NUM_EVAL})")
print(f"Fine-Tuned Model: {ft_acc:.1f}% exact match ({int(ft_acc*NUM_EVAL/100)}/{NUM_EVAL})")
print(f"Improvement:      +{ft_acc - base_acc:.1f}%")

# Show examples where fine-tuned model improved
print("\n--- Examples Where Fine-Tuned Won ---")
count = 0
for i in range(NUM_EVAL):
    if ft_results[i]["match"] and not base_results[i]["match"] and count < 5:
        count += 1
        print(f"\nExample {count}:")
        print(f"  Question:     {base_results[i]['question'][:150]}...")
        print(f"  Ground Truth: {base_results[i]['gt'][:150]}")
        print(f"  Base Output:  {base_results[i]['pred'][:150]}")
        print(f"  FT Output:    {ft_results[i]['pred'][:150]}")


In [ ]:
!pip install nltk

In [ ]:

# ============================================================
# CELL 7B: Enhanced Evaluation (run after Cell 7)
# ============================================================
# !pip install nltk

import json
import re
from pathlib import Path
from collections import defaultdict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

DATA_DIR = Path("data")

# ---- Load test samples with complexity info ----
from datasets import load_dataset
ds = load_dataset("gretelai/synthetic_text_to_sql")
test_indices = ds['test'].shuffle(seed=42).select(range(500, 1000))

# ---- Metrics ----
def normalize_sql(sql):
    sql = sql.lower().strip().rstrip(";") + ";"
    sql = re.sub(r'\s+', ' ', sql)
    return sql

def keyword_overlap(pred, gt):
    """Check if key SQL keywords/structure match."""
    pred_tokens = set(normalize_sql(pred).split())
    gt_tokens = set(normalize_sql(gt).split())
    if not gt_tokens:
        return 0.0
    return len(pred_tokens & gt_tokens) / len(gt_tokens)

def calc_bleu(pred, gt):
    """Calculate BLEU score between predicted and ground truth SQL."""
    ref = normalize_sql(gt).split()
    hyp = normalize_sql(pred).split()
    smooth = SmoothingFunction().method1
    try:
        return sentence_bleu([ref], hyp, smoothing_function=smooth)
    except:
        return 0.0

# ---- Compute enhanced metrics ----
NUM_EVAL = min(50, len(base_results), len(ft_results))

# Per-sample metrics
base_bleu_scores = []
ft_bleu_scores = []
base_keyword_scores = []
ft_keyword_scores = []

# Per-complexity metrics
complexity_results = defaultdict(lambda: {
    "base_exact": 0, "ft_exact": 0,
    "base_bleu": [], "ft_bleu": [],
    "count": 0
})

for i in range(NUM_EVAL):
    gt = base_results[i]["gt"]
    base_pred = base_results[i]["pred"]
    ft_pred = ft_results[i]["pred"]
    complexity = test_indices[i]["sql_complexity"]

    # BLEU
    b_bleu = calc_bleu(base_pred, gt)
    f_bleu = calc_bleu(ft_pred, gt)
    base_bleu_scores.append(b_bleu)
    ft_bleu_scores.append(f_bleu)

    # Keyword overlap
    base_keyword_scores.append(keyword_overlap(base_pred, gt))
    ft_keyword_scores.append(keyword_overlap(ft_pred, gt))

    # By complexity
    c = complexity_results[complexity]
    c["count"] += 1
    c["base_exact"] += int(base_results[i]["match"])
    c["ft_exact"] += int(ft_results[i]["match"])
    c["base_bleu"].append(b_bleu)
    c["ft_bleu"].append(f_bleu)

# ---- Print Results ----
print("=" * 70)
print("COMPREHENSIVE EVALUATION RESULTS")
print("=" * 70)

avg_base_bleu = sum(base_bleu_scores) / len(base_bleu_scores)
avg_ft_bleu = sum(ft_bleu_scores) / len(ft_bleu_scores)
avg_base_kw = sum(base_keyword_scores) / len(base_keyword_scores)
avg_ft_kw = sum(ft_keyword_scores) / len(ft_keyword_scores)
base_exact = sum(r["match"] for r in base_results[:NUM_EVAL])
ft_exact = sum(r["match"] for r in ft_results[:NUM_EVAL])

print(f"\n{'Metric':<25} {'Base Model':>15} {'Fine-Tuned':>15} {'Improvement':>15}")
print("-" * 70)
print(f"{'Exact Match':<25} {base_exact/NUM_EVAL*100:>14.1f}% {ft_exact/NUM_EVAL*100:>14.1f}% {(ft_exact-base_exact)/NUM_EVAL*100:>+14.1f}%")
print(f"{'BLEU Score':<25} {avg_base_bleu:>15.3f} {avg_ft_bleu:>15.3f} {avg_ft_bleu-avg_base_bleu:>+15.3f}")
print(f"{'Keyword Overlap':<25} {avg_base_kw*100:>14.1f}% {avg_ft_kw*100:>14.1f}% {(avg_ft_kw-avg_base_kw)*100:>+14.1f}%")

print(f"\n\n{'='*70}")
print("BREAKDOWN BY SQL COMPLEXITY")
print("=" * 70)
print(f"\n{'Complexity':<25} {'Count':>6} {'Base EM':>10} {'FT EM':>10} {'Base BLEU':>12} {'FT BLEU':>12}")
print("-" * 75)

for complexity in sorted(complexity_results.keys()):
    c = complexity_results[complexity]
    n = c["count"]
    if n == 0:
        continue
    base_em = c["base_exact"] / n * 100
    ft_em = c["ft_exact"] / n * 100
    base_b = sum(c["base_bleu"]) / n
    ft_b = sum(c["ft_bleu"]) / n
    print(f"{complexity:<25} {n:>6} {base_em:>9.1f}% {ft_em:>9.1f}% {base_b:>12.3f} {ft_b:>12.3f}")

# ---- Save enhanced results ----
enhanced_results = {
    "overall": {
        "base_exact_match": base_exact / NUM_EVAL * 100,
        "ft_exact_match": ft_exact / NUM_EVAL * 100,
        "base_bleu": avg_base_bleu,
        "ft_bleu": avg_ft_bleu,
        "base_keyword_overlap": avg_base_kw * 100,
        "ft_keyword_overlap": avg_ft_kw * 100,
    },
    "by_complexity": {
        k: {
            "count": v["count"],
            "base_exact_pct": v["base_exact"] / v["count"] * 100 if v["count"] > 0 else 0,
            "ft_exact_pct": v["ft_exact"] / v["count"] * 100 if v["count"] > 0 else 0,
            "base_bleu": sum(v["base_bleu"]) / v["count"] if v["count"] > 0 else 0,
            "ft_bleu": sum(v["ft_bleu"]) / v["count"] if v["count"] > 0 else 0,
        }
        for k, v in complexity_results.items()
    }
}

with open("eval_results_enhanced.json", "w") as f:
    json.dump(enhanced_results, f, indent=2)

print("\nEnhanced results saved to eval_results_enhanced.json")

In [ ]:
# ============================================================
# CELL 8: Save Results
# ============================================================
import json

examples = []
for i in range(NUM_EVAL):
    if ft_results[i]["match"] and not base_results[i]["match"] and len(examples) < 5:
        examples.append({
            "question": base_results[i]["question"][:300],
            "ground_truth": base_results[i]["gt"],
            "base_output": base_results[i]["pred"][:300],
            "finetuned_output": ft_results[i]["pred"][:300]
        })

eval_results = {
    "base_accuracy": base_acc,
    "finetuned_accuracy": ft_acc,
    "improvement": ft_acc - base_acc,
    "num_samples": NUM_EVAL,
    "examples": examples
}
with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Results saved to eval_results.json")

In [ ]:
# ============================================================
# CELL 9: Gradio Demo (using Ollama for base, MLX for fine-tuned)
# ============================================================
import gradio as gr
import requests
from mlx_lm import load, generate

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

FUSED_DIR = "fused_model"

# Load fine-tuned model
print("Loading fine-tuned model for demo...")
ft_model, ft_tokenizer = load(FUSED_DIR)

def query_base_ollama(prompt, system=SYSTEM_PROMPT):
    """Query base model via Ollama."""
    try:
        resp = requests.post("http://localhost:11434/api/generate", json={
            "model": "qwen2.5:3b",
            "prompt": prompt,
            "system": system,
            "stream": False,
            "options": {"temperature": 0.1, "num_predict": 300}
        })
        return resp.json().get("response", "ERROR")
    except Exception as e:
        return f"Error connecting to Ollama: {e}"

def query_finetuned_mlx(prompt, system=SYSTEM_PROMPT):
    """Query fine-tuned model via MLX."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]
    p = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate(ft_model, ft_tokenizer, prompt=p, max_tokens=300)

def generate_sql(schema, question, model_choice):
    prompt = f"""Database Schema:
{schema}

Question: {question}"""

    if model_choice == "Fine-Tuned":
        return query_finetuned_mlx(prompt)
    else:
        return query_base_ollama(prompt)

demo = gr.Interface(
    fn=generate_sql,
    inputs=[
        gr.Textbox(
            label="Database Schema",
            placeholder="CREATE TABLE employees (id INT, name VARCHAR(100), salary DECIMAL);",
            lines=5
        ),
        gr.Textbox(
            label="Natural Language Question",
            placeholder="What is the average salary?",
            lines=2
        ),
        gr.Radio(
            choices=["Fine-Tuned", "Base Model"],
            label="Model",
            value="Fine-Tuned"
        )
    ],
    outputs=gr.Textbox(label="Generated SQL + Explanation", lines=8),
    title="Text-to-SQL Generator",
    description="Generate SQL queries from natural language. Compare base vs fine-tuned model.",
    examples=[
        [
            "CREATE TABLE employees (id INT, name VARCHAR(100), department VARCHAR(50), salary DECIMAL(10,2));\nCREATE TABLE departments (id INT, name VARCHAR(50), budget DECIMAL(12,2));",
            "What is the average salary by department?",
            "Fine-Tuned"
        ],
        [
            "CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);\nCREATE TABLE customers (id INT, name VARCHAR(100), city VARCHAR(50));",
            "Find the top 5 customers by total order amount",
            "Fine-Tuned"
        ]
    ]
)

demo.launch(share=False)